# # PRE_09 — Horas y millas por conductor, asistido por ChatGPT
#
# **Qué se hace:** el mismo análisis de PRE_08 (resumen por conductor + top 10 por millas), pero
# apoyándose en un modelo de lenguaje para escribir el código. Aquí queda registrado el proceso:
# los prompts usados y lo que hubo que corregir a mano de cada respuesta. El código final es la
# versión ya revisada, en pandas.
#
# **Entregables** (`submission/`): `summary.csv` y `top10_drivers.png`.
#
# ## Prompts usados
# 1. *"Tengo drivers.csv (driverId, name, ssn, location, certified, wage-plan) y timesheet.csv
#    (driverId, week, hours-logged, miles-logged). Quiero un resumen por conductor con total y
#    promedio de horas y millas."*
# 2. *"Agrégale millas por hora y ordénalo de mayor a menor."*
# 3. *"Haz un gráfico de barras horizontales con los 10 primeros."*
#
# ## Qué hubo que corregir de las respuestas
# - Proponía un `merge` interno; se cambió a `left` para no perder conductores sin registros.
# - Incluía `ssn` y `location` en el resumen: son identificadores directos y el reporte es agregado.
# - No fijaba el backend de matplotlib (`Agg`), así que fallaba fuera de un notebook.
# - Dividía millas entre horas sin proteger el caso de horas en cero.
# - Los valores del gráfico no estaban rotulados sobre las barras y el mayor quedaba abajo.

In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# Funciona tanto desde la carpeta de la actividad como desde notebooks/ (Jupyter)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
SUBMISSION = ROOT / "submission"
SUBMISSION.mkdir(exist_ok=True)

drivers = pd.read_csv(ROOT / "data" / "drivers.csv")
timesheet = pd.read_csv(ROOT / "data" / "timesheet.csv")
print(drivers.shape, timesheet.shape)
print("Nulos:", int(drivers.isna().sum().sum()), int(timesheet.isna().sum().sum()))

(34, 6) (1768, 4)
Nulos: 0 0


# ## Resumen por conductor (versión corregida)

In [2]:
agg = timesheet.groupby("driverId").agg(
    weeks=("week", "count"),
    total_hours=("hours-logged", "sum"),
    total_miles=("miles-logged", "sum"),
    mean_hours=("hours-logged", "mean"),
    mean_miles=("miles-logged", "mean"),
)

# Left join desde drivers: un conductor sin registros no debe desaparecer.
summary = drivers.merge(agg, on="driverId", how="left")
counts = ["weeks", "total_hours", "total_miles"]
summary[counts] = summary[counts].fillna(0).astype(int)
summary[["mean_hours", "mean_miles"]] = summary[["mean_hours", "mean_miles"]].round(1)
# horas = 0 produce inf/NaN: se reemplaza por 0
summary["miles_per_hour"] = (
    (summary["total_miles"] / summary["total_hours"].replace(0, float("nan"))).round(2).fillna(0)
)

columns = ["driverId", "name", "certified", "wage-plan", "weeks", "total_hours",
           "total_miles", "mean_hours", "mean_miles", "miles_per_hour"]  # sin ssn ni location
summary = summary[columns].sort_values("total_miles", ascending=False).reset_index(drop=True)

assert len(summary) == len(drivers)
assert summary.total_hours.sum() == timesheet["hours-logged"].sum()
assert summary.total_miles.sum() == timesheet["miles-logged"].sum()
assert not {"ssn", "location"} & set(summary.columns)
summary.to_csv(SUBMISSION / "summary.csv", index=False)
summary.head(10)

,driverId,name,certified,wage-plan,weeks,total_hours,total_miles,mean_hours,mean_miles,miles_per_hour
0,11,Jamie Engesser,N,miles,52,3642,179300,70.0,3448.1,49.23
1,10,George Vetticaden,N,miles,52,3232,147150,62.2,2829.8,45.53
2,33,Sridhara Sabbella,Y,hours,52,2759,139285,53.1,2678.6,50.48
3,25,Jean-Philippe Playe,Y,hours,52,2723,139180,52.4,2676.5,51.11
4,39,David Kaiser,Y,hours,52,2745,138788,52.8,2669.0,50.56
5,15,Rohit Bakshi,Y,hours,52,2734,138750,52.6,2668.3,50.75
6,35,Emil Siemes,Y,hours,52,2728,138727,52.5,2667.8,50.85
7,21,Jeff Markham,Y,hours,52,2751,138719,52.9,2667.7,50.42
8,41,Greg Phillips,Y,hours,52,2723,138407,52.4,2661.7,50.83
9,29,Teddy Choi,Y,hours,52,2760,138255,53.1,2658.8,50.09


# ## Top 10 por millas

In [3]:
top = summary.nlargest(10, "total_miles").iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(top["name"], top["total_miles"], color="#4C78A8")
ax.set_xlabel("Millas recorridas en el año")
ax.set_title("Top 10 conductores por millas recorridas")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_xlim(0, top["total_miles"].max() * 1.12)
for bar, value in zip(bars, top["total_miles"]):
    ax.text(value * 1.01, bar.get_y() + bar.get_height() / 2, f"{value:,}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(SUBMISSION / "top10_drivers.png", dpi=150)
plt.show()

/var/folders/gj/yjfyzlqd4fxcmlnj64g5s9bw0000gn/T/ipykernel_6254/1664987035.py:14: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


# ## Hallazgos y verificación

In [4]:
print(summary.groupby("wage-plan")[["total_miles", "total_hours", "miles_per_hour"]].mean().round(2))
print(summary.groupby("certified")["total_miles"].mean().round(0))

# Verificación independiente: se recalcula con SQL y debe dar lo mismo que pandas
import sqlite3
con = sqlite3.connect(":memory:")
timesheet.to_sql("timesheet", con, index=False)
check = pd.read_sql_query(
    'SELECT driverId, SUM("miles-logged") AS m, SUM("hours-logged") AS h FROM timesheet GROUP BY driverId', con
).set_index("driverId")
con.close()
merged = summary.set_index("driverId").join(check)
assert (merged.total_miles == merged.m).all() and (merged.total_hours == merged.h).all()
print("Cruce pandas vs SQL: OK")

           total_miles  total_hours  miles_per_hour
wage-plan                                          
hours        137383.97      2728.97           50.35
miles        163225.00      3437.00           47.38
certified
N    163225.0
Y    137384.0
Name: total_miles, dtype: float64
Cruce pandas vs SQL: OK


# - Resultado idéntico al de PRE_08 (SQL): 34 conductores, mismo total de horas y millas; el plan `miles`
#   recorre más pero con menos millas por hora (47,4 vs 50,3).
# - Aunque el asistente dio una base útil, los tres errores de fondo (join interno, datos personales
#   en el reporte, división sin proteger) solo se detectan revisando el resultado.
# - La verificación cruzada contra una consulta SQL independiente confirma que las cifras son correctas.